## 前処理07: swing_result のクラス統合 (9→3)

本ファイルは以下の処理を行うコードである。

1. `/workspace/datasets/statcast-customized-tmp/preprocess_06` のデータセットを読みだす。

2. `swing_result` の9クラスを3クラスに統合する。

| 旧クラス | 旧ラベル名 | → 新クラス | 新ラベル名 |
| --- | --- | --- | --- |
| 0 | foul | 0 | foul |
| 3 | foul_tip | 0 | foul |
| 5 | foul_bunt | 0 | foul |
| 7 | bunt_foul_tip | 0 | foul |
| 8 | foul_pitchout | 0 | foul |
| 1 | hit_into_play | 1 | hit_into_play |
| 2 | swinging_strike | 2 | miss |
| 4 | swinging_strike_blocked | 2 | miss |
| 6 | missed_bunt | 2 | miss |

3. 統合後のデータを `/workspace/datasets/statcast-customized-tmp/preprocess_07` に保存する。

4. stats ファイルを更新する。

5. 保存後、`/workspace/datasets/statcast-customized/` にコピーして運用に反映する。

In [ ]:
import os
import shutil
import pandas as pd

In [ ]:
load_root_path = '/workspace/datasets/statcast-customized-tmp/preprocess_06/'
save_root_path = '/workspace/datasets/statcast-customized-tmp/preprocess_07/'

# 旧クラスラベル → 新クラスラベル のマッピング
# 0:foul, 3:foul_tip, 5:foul_bunt, 7:bunt_foul_tip, 8:foul_pitchout → 0:foul
# 1:hit_into_play → 1:hit_into_play
# 2:swinging_strike, 4:swinging_strike_blocked, 6:missed_bunt → 2:miss
swing_result_remap = {
    0: 0,  # foul → foul
    1: 1,  # hit_into_play → hit_into_play
    2: 2,  # swinging_strike → miss
    3: 0,  # foul_tip → foul
    4: 2,  # swinging_strike_blocked → miss
    5: 0,  # foul_bunt → foul
    6: 2,  # missed_bunt → miss
    7: 0,  # bunt_foul_tip → foul
    8: 0,  # foul_pitchout → foul
}

new_swing_result_stats = [
    {'class_label': 0, 'swing_result': 'foul', 'description': 'foul + foul_tip + foul_bunt + bunt_foul_tip + foul_pitchout'},
    {'class_label': 1, 'swing_result': 'hit_into_play', 'description': 'hit_into_play'},
    {'class_label': 2, 'swing_result': 'miss', 'description': 'swinging_strike + swinging_strike_blocked + missed_bunt'},
]

In [ ]:
def get_parquet_file_path_list(root_path):
    file_path_list = []
    for root, dirs, files in os.walk(root_path):
        for file in files:
            if file.endswith('.parquet'):
                file_path_list.append(os.path.join(root, file))
    file_path_list.sort()
    return file_path_list

In [ ]:
# 対象ファイル一覧を取得
parquet_file_path_list = get_parquet_file_path_list(load_root_path)
print(f'対象ファイル数: {len(parquet_file_path_list)}')
for f in parquet_file_path_list[:5]:
    print(f'  {f}')
print('  ...')

In [ ]:
# 1ファイルで事前確認
test_path = parquet_file_path_list[0]
df_test = pd.read_parquet(test_path)

print('変換前の swing_result 分布:')
print(df_test['swing_result'].value_counts(dropna=False).sort_index())
print()

# 変換テスト
sr = df_test['swing_result'].copy()
sr_mapped = sr.map(swing_result_remap)
print('変換後の swing_result 分布:')
print(sr_mapped.value_counts(dropna=False).sort_index())

In [ ]:
# 全ファイルで swing_result を統合して preprocess_07 に保存
os.makedirs(save_root_path, exist_ok=True)

total_counts = {0: 0, 1: 0, 2: 0}  # 新クラスごとの件数集計

for parquet_path in parquet_file_path_list:
    fname = os.path.basename(parquet_path)
    df = pd.read_parquet(parquet_path)

    # swing_result のリマッピング（<NA> はそのまま保持）
    df['swing_result'] = df['swing_result'].map(swing_result_remap)

    save_path = os.path.join(save_root_path, fname)
    df.to_parquet(save_path, index=False)

    # 集計
    counts = df['swing_result'].value_counts(dropna=True)
    for cls, cnt in counts.items():
        total_counts[cls] = total_counts.get(cls, 0) + cnt

    print(f'OK: {fname} ({len(df)} rows)')

print()
print('全体の新クラス分布:')
for cls, cnt in sorted(total_counts.items()):
    print(f'  {cls}: {cnt:,}')

In [ ]:
# 結果の検証
saved_files = get_parquet_file_path_list(save_root_path)
print(f'保存ファイル数: {len(saved_files)}')

df_verify = pd.read_parquet(saved_files[0])
print(f'カラム数: {len(df_verify.columns)}')
print()
print('swing_result 分布 (検証):')
print(df_verify['swing_result'].value_counts(dropna=False).sort_index())
print()
# 旧クラス (3以上) が存在しないことを確認
sr_valid = df_verify['swing_result'].dropna()
assert sr_valid.max() <= 2, f'ERROR: 旧クラスが残っています (max={sr_valid.max()})'
assert sr_valid.min() >= 0, f'ERROR: 不正な値 (min={sr_valid.min()})'
print('検証OK: swing_result は 0, 1, 2 のみ')

In [ ]:
# stats ファイルを更新
stats_save_dir = '/workspace/datasets/statcast-customized-tmp/preprocess_07/stats'
os.makedirs(stats_save_dir, exist_ok=True)

# 既存の stats をコピー
stats_src_dir = '/workspace/datasets/statcast-customized/stats'
for fname in os.listdir(stats_src_dir):
    if fname.startswith('stats_') and fname != 'stats_swing_result.csv' and fname != 'stats_all.csv':
        shutil.copy2(os.path.join(stats_src_dir, fname), os.path.join(stats_save_dir, fname))

# 新しい swing_result stats を作成
stats_sr = pd.DataFrame([
    {'class_label': 0, 'swing_result': 'foul', 'count': total_counts[0]},
    {'class_label': 1, 'swing_result': 'hit_into_play', 'count': total_counts[1]},
    {'class_label': 2, 'swing_result': 'miss', 'count': total_counts[2]},
])
stats_sr.to_csv(os.path.join(stats_save_dir, 'stats_swing_result.csv'), index=False)
print('swing_result stats:')
print(stats_sr)

# stats_all.csv を再構築
stats_all_parts = []
for fname in sorted(os.listdir(stats_save_dir)):
    if fname == 'stats_all.csv' or not fname.startswith('stats_'):
        continue
    feature_name = fname.replace('stats_', '').replace('.csv', '')
    tmp = pd.read_csv(os.path.join(stats_save_dir, fname))
    value_col = [c for c in tmp.columns if c not in ['class_label', 'count']][0]
    tmp = tmp.rename(columns={value_col: 'value'})
    tmp.insert(0, 'feature', feature_name)
    stats_all_parts.append(tmp[['feature', 'class_label', 'value', 'count']])

stats_all = pd.concat(stats_all_parts, ignore_index=True)
stats_all.to_csv(os.path.join(stats_save_dir, 'stats_all.csv'), index=False)
print()
print(f'stats_all.csv saved ({len(stats_all)} rows)')

### 運用データへの反映

`preprocess_07` のデータと stats を `/workspace/datasets/statcast-customized/` にコピーする。

In [ ]:
# preprocess_07 → statcast-customized にコピー
final_data_dir = '/workspace/datasets/statcast-customized/data'
final_stats_dir = '/workspace/datasets/statcast-customized/stats'

# データファイルのコピー
for fpath in saved_files:
    fname = os.path.basename(fpath)
    shutil.copy2(fpath, os.path.join(final_data_dir, fname))
print(f'データコピー完了: {len(saved_files)} ファイル → {final_data_dir}')

# stats ファイルのコピー
for fname in os.listdir(stats_save_dir):
    shutil.copy2(os.path.join(stats_save_dir, fname), os.path.join(final_stats_dir, fname))
print(f'statsコピー完了 → {final_stats_dir}')